# Exercice 3 - Modèle de Merton

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve

## Question 1 - Demonstration

**Premiere equation :** Dans Merton, l'equity c'est un call sur la valeur des actifs V avec strike D (la dette). Donc par Black-Scholes : S = V*N(d1) - D*exp(-rT)*N(d2). C'est direct.

**Deuxieme equation :** On applique Itô à S = f(V,t). Le terme en dW donne sigma_S * S = (dS/dV) * sigma_V * V. Or le delta du call c'est N(d1), donc on a sigma_S = sigma_V * V/S * N(d1).

**Unicité :** La premiere eq est croissante en V (prix du call croissant) et la seconde donne une relation monotone entre V et sigma_V. Le systeme a donc une seule solution.

## Question 2 - Resolution numérique

In [ ]:
S = 25
D = 98.78
r = 0.05
T = 1
sig_S = 0.30

In [ ]:
def systeme(x):
    V, sig_V = x
    
    d1 = (np.log(V/D) + (r + 0.5*sig_V**2)*T) / (sig_V * np.sqrt(T))
    d2 = d1 - sig_V * np.sqrt(T)
    
    eq1 = V * norm.cdf(d1) - D * np.exp(-r*T) * norm.cdf(d2) - S
    eq2 = sig_V * V/S * norm.cdf(d1) - sig_S
    
    return [eq1, eq2]

In [ ]:
# point de depart : V ~ S + dette actualisée
V0 = S + D * np.exp(-r*T)
sig0 = sig_S * S / V0

sol = fsolve(systeme, [V0, sig0])
V_star, sig_V_star = sol

print(f"V = {V_star:.2f}")
print(f"sigma_V = {sig_V_star*100:.2f}%")

In [ ]:
# on verifie que le systeme est bien resolu
res = systeme([V_star, sig_V_star])
print(f"Résidus : {res[0]:.2e}, {res[1]:.2e}")

# proba de defaut
d2 = (np.log(V_star/D) + (r - 0.5*sig_V_star**2)*T) / (sig_V_star*np.sqrt(T))
print(f"\nPD (risk-neutral) = {norm.cdf(-d2)*100:.2f}%")

La valeur des actifs est ~119, soit bien plus que l'equity (25) car V = equity + dette. La vol des actifs (~6%) est très inférieure à celle de l'equity (30%) à cause de l'effet de levier : la dette amplifie la vol.